# Partial Freezing of a Multilingual Model for Chinese PoS Tagging

Compare full fine-tuning with freezing embeddings and the first three of DistilBERT’s six layers on a fixed Universal Dependencies Chinese GSD sample. The metric is word-level UPOS accuracy on the official dev split.

## 1. Install dependencies

Colab already provides PyTorch. This installs the Hugging Face model library and the small download dependency.

In [ ]:
%pip install -q 'transformers>=4.45,<5' requests

## 2. Get the code and check the runtime

If you opened the notebook from GitHub, this checks out the companion script. If you cloned the repository and opened the notebook inside it, it uses those files.

In [ ]:
from pathlib import Path
import os, platform, subprocess, torch
if not Path('run_experiment.py').exists():
    subprocess.run(['git', 'clone', 'https://github.com/billchenziang33/csc394-pos-partial-freezing.git'], check=True)
    os.chdir('csc394-pos-partial-freezing')
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert Path('run_experiment.py').exists()

## 3. Run both conditions

Each condition starts from a fresh pretrained checkpoint. They use the same data sample, seed, batch size, learning rate, and epochs.

In [ ]:
%run run_experiment.py --max-train 600 --max-dev 150 --epochs 3 --batch-size 16 --learning-rate 2e-5 --seed 42

## 4. Read the measured scores

Do not interpret these values unless the run above completed successfully.

In [ ]:
import json
from pathlib import Path
results = json.loads(Path('results.json').read_text(encoding='utf-8'))
for row in results['results']:
    print(f"{row['condition']}: {row['dev_word_accuracy']:.4%} ({row['correct_words']}/{row['dev_words']} words); trainable parameters {row['trainable_parameters']:,}/{row['total_parameters']:,}")
difference = results['results'][1]['dev_word_accuracy'] - results['results'][0]['dev_word_accuracy']
print(f'Partial freezing minus full fine-tuning: {difference:+.4%}')

## 5. Interpretation

Compare accuracy, training loss, and trainable parameter count. This is a small one-seed experiment, so a difference may be sampling noise; it does not establish a general advantage. Explain whether the result fits the idea that fixed lower layers can help or limit adaptation. Repeat multiple seeds and languages before making a broader claim.